# tally-cam · RF-DETR small 미세조정 (Colab GPU)

사람 점 라벨에서 만든 의사 상자(`bench/pseudo/out/*.json`)로 RF-DETR small을 미세조정하고, **학습에 쓰지 않은 카메라**로 검출 → 세기(A8) → 채점까지 합니다.

**준비 (한 번만)**: Google Drive에 `tally-cam/samples/` 폴더를 만들고 PC의 `samples` 폴더에서 아래 5개 영상을 올리세요.
`dongtan_night_01.mp4, wonji_am_01.mp4, wonji_am_02.mp4, dalraenae1_am_01.mp4, songpa_am_01.mp4`
(테스트 영상 `dongtan_night_02.mp4`는 올리지 마세요.)

**런타임**: 메뉴 > 런타임 > 런타임 유형 변경 > GPU(T4 이상).

In [ ]:
# 1. GPU 확인 + Drive 연결
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/tally-cam'

In [ ]:
# 2. 코드 받기 + 설치 (3~5분)
!git clone -q https://github.com/flytothejy/tally-cam.git /content/tally-cam
%cd /content/tally-cam
!pip install -q "rfdetr[train,loggers]==1.11.2" ultralytics opencv-python-headless scipy
!mkdir -p samples && cp $DRIVE/samples/*.mp4 samples/ && ls -la samples

In [ ]:
# 3. 영상에서 학습 이미지 뽑기 (의사 상자 JSON 재사용, 2~3분)
!python bench/pseudo/pseudo_boxes.py --all --reuse --out bench/pseudo/out --export data/pseudo_coco

In [ ]:
# 4. 카메라 하나씩 빼고 학습 + 그 카메라로 채점
#    T4 기준 겹당 30~40분 예상. 시간이 부족하면 FOLDS를 줄이세요(약한 곳부터: songpa, dongtan-jct).
FOLDS = ['songpa', 'dongtan-jct', 'wonji', 'dalraenae1']
EPOCHS = 12
for cam in FOLDS:
    !python bench/train/train_rfdetr.py --data data/pseudo_coco --hold {cam} --epochs {EPOCHS} --bs 8 --accum 2 --out runs --eval
    !mkdir -p $DRIVE/runs/wo_{cam} && cp runs/wo_{cam}/checkpoint_best_total.pth runs/wo_{cam}/metrics.csv $DRIVE/runs/wo_{cam}/
    !mkdir -p $DRIVE/preds && cp -r bench/preds/FT-rfdetrS-wo_{cam} $DRIVE/preds/

In [ ]:
# 5. 요약: 빠진 카메라 클립 점수 (기준 A8 = 미세조정 전 RF-DETR + 같은 세기 규칙)
import json, glob, sys
sys.path.insert(0, 'bench')
from score import score_clip, total
ds = json.load(open('bench/dataset.json'))
cam = {c['id']: c['camera'] for c in ds['clips']}
for c in ds['clips']:
    if c['split'] != 'dev' or not c['gt']: continue
    p = f"bench/preds/FT-rfdetrS-wo_{c['camera']}/{c['id']}.json"
    if not glob.glob(p): continue
    t = total(list(score_clip(c['gt'], p)['dirs'].values()))
    print(f"{c['id']:18} F1 {t['f1']*100:.1f}  재현율 {t['recall']*100:.1f}  정밀도 {t['precision']*100:.1f}  놓침 {t['fn']}  헛셈 {t['fp']}")
print('\n결과 폴더(Drive): tally-cam/preds, tally-cam/runs — preds 폴더를 Claude에게 보내 주세요.')